# cNMF Evaluation Pipeline Tutorial

This notebook demonstrates how to evaluate gene programs discovered by cNMF. It mirrors `Slurm_Version/cNMF_evaluation_pipeline.py` so that interactive runs produce the same outputs (filenames + contents) as SLURM runs.

**Prerequisites**: You should have already run cNMF inference (prepare, factorize, combine, consensus) and saved results as `.h5mu` MuData files. Each `.h5mu` must contain the cNMF program modality (`prog_key`) and — for perturbation association — `guide_assignment`, `guide_names`, and `guide_targets` in `mdata[prog_key]`.

## Pipeline Steps

| Step | Description |
|------|-------------|
| **Step 1. Set Up** | Imports, I/O paths (`out_dir`, `run_name`), resources (GWAS, normalized counts), MuData keys, and the (K, threshold) grid |
| **Step 2. Load Non-targeting Guides (Optional)** | Either read a guide annotation TSV or fall back to `guide_annotation_key` (default `["non-targeting"]`) as the reference set for perturbation tests |
| **Step 3. Load Resources for Explained Variance** | Build the cNMF object and load the normalized expression matrix |
| **Step 4. Optional column renaming** | Helper to rename `mdata[data_key].var_names` from Ensembl IDs to gene symbols (only run when needed) |
| **Step 5. Evaluation Loop** | For each `(K, threshold)` combination, load the `.h5mu` and run each enabled metric |

### Evaluation Sub-steps (inside Step 5) — order matches the SLURM `.py`

| Sub-step | Output File | Description |
|----------|-------------|-------------|
| **5a. Categorical Association** | `{K}_categorical_association_results.txt`, `{K}_categorical_association_posthoc.txt` | Kruskal-Wallis + Dunn's posthoc — do program scores differ across categories? |
| **5b. Perturbation Association** | `{K}_perturbation_association_results_{sample}.txt` | Mann-Whitney U per sample — perturbed cells vs non-targeting controls |
| **5c. Gene-Set Enrichment (Reactome)** | `{K}_geneset_enrichment.txt` | Fisher test of top `n_top` program genes vs Reactome 2022 |
| **5d. Gene-Set Enrichment (GO BP)** | `{K}_GO_term_enrichment.txt` | Fisher test of top `n_top` program genes vs GO Biological Process 2023 |
| **5e. Trait Enrichment** | `{K}_trait_enrichment.txt` | Fisher test of top program genes vs Open Targets L2G GWAS genes |
| **5f. Motif Enrichment** (optional / commented out) | `cNMF_{class}_pearson_topn2000_{sample}_motif_*.txt` | Pearson correlation of motif counts vs program scores for enhancer/promoter loci |
| **5g. Explained Variance** | `{K}_Explained_Variance.txt`, `{K}_Explained_Variance_Summary.txt` | Variance of normalized expression captured by each consensus program |

## Expected Output Structure

```
{out_dir}/{run_name}/
├── Inference/                                           # (read-only inputs from the inference step)
│   ├── cnmf_tmp/
│   │   └── Inference.norm_counts.h5ad                   # Used for explained variance
│   └── adata/
│       └── cNMF_{K}_{thresh}.h5mu                       # Input MuData per (K, threshold)
│
└── Evaluation/
    └── {K}_{thresh}/                                    # One folder per (K, threshold)
        ├── {K}_categorical_association_results.txt
        ├── {K}_categorical_association_posthoc.txt
        ├── {K}_perturbation_association_results_{sample}.txt
        ├── {K}_geneset_enrichment.txt
        ├── {K}_GO_term_enrichment.txt
        ├── {K}_trait_enrichment.txt
        ├── {K}_Explained_Variance.txt
        ├── {K}_Explained_Variance_Summary.txt
        └── (optional) motif files: cNMF_{class}_pearson_topn2000_{sample}_motif_*.txt
```

### Notes

- `{K}` = number of components (e.g., 30, 50). `{thresh}` = density threshold with `.` → `_` (e.g., `0_4`, `2_0`).
- `{sample}` = unique values of `categorical_key` (e.g., condition labels).
- Perturbation association requires `guide_assignment`/`guide_names`/`guide_targets` to already live inside the `.h5mu` (these are written during inference). The notebook does **not** load a separate guide AnnData.
- If `guide_annotation_path` extracts guide *names* (e.g. `non-targeting_00014`), `compute_perturbation_association` — which expects target *group* names — will silently get an empty reference set. Prefer `guide_annotation_key=["non-targeting"]`.
- Explained variance requires `Inference/cnmf_tmp/Inference.norm_counts.h5ad`.
- Motif enrichment is left commented out (matches `.py`); it requires HOCOMOCO motifs, hg38 FASTA, and scE2G loci files.
- These outputs are the input to the downstream **Calibration** and **Interpretation** pipelines.


In [ ]:
import os
import sys
import yaml

import h5py
import mudata as mu
import pandas as pd
import numpy as np
import cnmf
import scanpy as sc

# Change path to wherever you have the repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')

from Stage2_Evaluation.A_Metrics.src import (
    compute_categorical_association,
    compute_geneset_enrichment,
    compute_trait_enrichment,
    compute_perturbation_association,
    compute_explained_variance,
    compute_motif_enrichment,
)


## Step 1. Set Up

In [ ]:
# ── IO ──
out_dir = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result"
run_name = "030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50"

# Values of K (number of programs) and density thresholds to evaluate
K = [30, 50, 60, 80, 100, 200, 250, 300]
sel_threshs = [0.4, 0.8, 2.0]

# ── Which metrics to run (set to False to skip) ──
Perform_categorical = True
Perform_perturbation = True
Perform_geneset = True
Perform_trait = True
Perform_explained_variance = True
Perform_motif = False

# ── Resource files ──
# Normalized counts produced by cNMF prepare (inside Inference/cnmf_tmp/)
X_normalized_path = f"{out_dir}/{run_name}/Inference/cnmf_tmp/Inference.norm_counts.h5ad"

# Open Targets GWAS data for trait enrichment
gwas_data_path = "/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage2_Evaluation/Resources/OpenTargets_L2G_Filtered.csv.gz"

# (Optional) Guide annotation file — set to None to fall back to `guide_annotation_key` below
guide_annotation_path = None  # e.g. "/path/to/guide_metadata.tsv"


In [ ]:
# ── MuData keys (must match what inference wrote) ──
data_key = "rna"                         # RNA expression modality
prog_key = "cNMF"                        # cNMF program modality
categorical_key = "sample"               # obs column with sample/condition labels
gene_names_key = "symbol"                # var column with gene symbols (used in enrichment)

guide_names_key = "guide_names"
guide_targets_key = "guide_targets"
guide_assignment_key = "guide_assignment"
guide_annotation_key = ["non-targeting"] # used when guide_annotation_path is None

# ── Misc ──
organism = "human"                       # "human" or "mouse"
FDR_method = "StoreyQ"
n_top = 300                              # top-N program genes for enrichment
use_cache = False                        # load gene-set libraries from Resources/ cache if available
reassign_name = False                    # call _reassign_name() before running metrics

# ── Guards (mirror the argparse checks in the .py) ──
if Perform_trait and gwas_data_path is None:
    raise ValueError("gwas_data_path is required when Perform_trait is True")


## Step 2. Load Non-targeting Guides (Optional)

If `guide_annotation_path` is provided, load non-targeting *guide names* from the TSV. Otherwise fall back to `guide_annotation_key`.

> **Heads-up**: `compute_perturbation_association` expects target **group** names (e.g. `"non-targeting"`), not individual guide names (e.g. `"non-targeting_00014"`). Passing a TSV that gives you guide names produces a silently empty reference set. Prefer the key-based path (default).


In [ ]:
# list of non-targeting guides / target groups
if guide_annotation_path is not None:
    df_target = pd.read_csv(guide_annotation_path, sep="\t", index_col=0)
    df_target_non = df_target[df_target["targeting"] == False]
    reference_targets = df_target_non.index.values.tolist()
else:
    reference_targets = guide_annotation_key


## Step 3. Load Resources for Explained Variance

In [ ]:
# Objects used in explained variance calculation
if Perform_explained_variance:
    cnmf_obj = cnmf.cNMF(output_dir=f"{out_dir}/{run_name}", name="Inference")
    X_norm = sc.read_h5ad(X_normalized_path)
    X = X_norm.X


## Step 4. Optional: reassign `var_names`

Only needed when `mdata[data_key].var_names` are Ensembl IDs and you want gene symbols (so enrichment libraries hit). Toggle via the `reassign_name` flag in Step 1.


In [ ]:
def _reassign_name(mdata, gene_names_key="symbol", data_key="rna"):
    mdata[data_key].var_names = mdata[data_key].var[gene_names_key].astype(str)


## Step 5. Evaluation Loop

In [ ]:
for sel_thresh in sel_threshs:
    thresh_str = str(sel_thresh).replace(".", "_")

    for k in K:
        print(f"\n{'='*60}")
        print(f"Evaluating K={k}, density_threshold={sel_thresh}")
        print(f"{'='*60}")

        output_folder = f"{out_dir}/{run_name}/Evaluation/{k}_{thresh_str}"
        os.makedirs(output_folder, exist_ok=True)

        # ── Load MuData ──
        mdata_path = f"{out_dir}/{run_name}/Inference/adata/cNMF_{k}_{thresh_str}.h5mu"
        mdata = mu.read(mdata_path)

        if reassign_name:
            _reassign_name(mdata, gene_names_key=gene_names_key, data_key=data_key)

        # ────────────────────────────────────────────────────────────
        # 5a. Categorical Association
        # ────────────────────────────────────────────────────────────
        if Perform_categorical:
            print("Running categorical association...")
            results_df, posthoc_df = compute_categorical_association(
                mdata,
                prog_key=prog_key,
                categorical_key=categorical_key,
                pseudobulk_key=None,
                test="dunn",
                n_jobs=-1,
                inplace=False,
            )
            results_df.to_csv(f"{output_folder}/{k}_categorical_association_results.txt", sep="\t", index=False)
            posthoc_df.to_csv(f"{output_folder}/{k}_categorical_association_posthoc.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5b. Perturbation Association
        #     Validates that reference_targets actually overlap the
        #     mdata guide_targets before running.
        # ────────────────────────────────────────────────────────────
        if Perform_perturbation:
            mdata_targets = set(mdata[prog_key].uns[guide_targets_key])
            matched_ref = mdata_targets.intersection(reference_targets)
            print(f"[K={k}] reference_targets overlap with mdata guide_targets: {matched_ref}")
            if len(matched_ref) == 0:
                raise ValueError(
                    "No reference_targets found in mdata guide_targets. "
                    f"reference_targets contains (e.g.) {list(reference_targets)[:3]}, "
                    f"but guide_targets contains group names (e.g.) {list(mdata_targets)[:3]}. "
                    "Use guide_annotation_key instead of guide_annotation_path."
                )

            print("Running perturbation association...")
            for samp in mdata[data_key].obs[categorical_key].unique():
                mdata_sub = mdata[mdata[data_key].obs[categorical_key] == samp]
                test_stats_df = compute_perturbation_association(
                    mdata_sub,
                    prog_key=prog_key,
                    collapse_targets=True,
                    pseudobulk=False,
                    reference_targets=reference_targets,
                    n_jobs=-1,
                    inplace=False,
                    FDR_method=FDR_method,
                )
                test_stats_df.to_csv(
                    f"{output_folder}/{k}_perturbation_association_results_{samp}.txt",
                    sep="\t",
                    index=False,
                )

        # ────────────────────────────────────────────────────────────
        # 5c. Gene-Set Enrichment (Reactome)
        # 5d. Gene-Set Enrichment (GO Biological Process)
        # ────────────────────────────────────────────────────────────
        if Perform_geneset:
            print("Running Reactome gene-set enrichment...")
            reactome_res = compute_geneset_enrichment(
                mdata,
                prog_key=prog_key,
                data_key=data_key,
                prog_name=None,
                organism=organism,
                library="Reactome_2022",
                method="fisher",
                database="enrichr",
                n_top=n_top,
                n_jobs=-1,
                inplace=False,
                user_geneset=None,
                use_loadings_gene=False,
                gene_names_key=gene_names_key,
                use_cache=use_cache,
            )
            reactome_res.to_csv(f"{output_folder}/{k}_geneset_enrichment.txt", sep="\t", index=False)

            print("Running GO Biological Process enrichment...")
            go_res = compute_geneset_enrichment(
                mdata,
                prog_key=prog_key,
                data_key=data_key,
                prog_name=None,
                organism=organism,
                library="GO_Biological_Process_2023",
                method="fisher",
                database="enrichr",
                n_top=n_top,
                n_jobs=-1,
                inplace=False,
                user_geneset=None,
                use_loadings_gene=False,
                gene_names_key=gene_names_key,
                use_cache=use_cache,
            )
            go_res.to_csv(f"{output_folder}/{k}_GO_term_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5e. Trait Enrichment (GWAS, Open Targets L2G)
        # ────────────────────────────────────────────────────────────
        if Perform_trait:
            print("Running GWAS trait enrichment...")
            trait_res = compute_trait_enrichment(
                mdata,
                gwas_data=gwas_data_path,
                prog_key=prog_key,
                prog_name=None,
                data_key=data_key,
                library="OT_GWAS",
                n_jobs=-1,
                inplace=False,
                key_column="trait_efos",
                gene_column="gene_name",
                method="fisher",
                n_top=n_top,
                use_loadings_gene=True,
                gene_names_key=gene_names_key,
            )
            trait_res.to_csv(f"{output_folder}/{k}_trait_enrichment.txt", sep="\t", index=False)

        # ────────────────────────────────────────────────────────────
        # 5f. Motif Enrichment (work in progress — kept commented to mirror the .py)
        # ────────────────────────────────────────────────────────────
        if Perform_motif:
            """
            fimo_thresh_enhancer = 1e-6
            fimo_thresh_promoter = 1e-4

            for samp in mdata[data_key].obs[categorical_key].unique():
                for class_, thresh in [("enhancer", fimo_thresh_enhancer),
                                       ("promoter", fimo_thresh_promoter)]:
                    loci_file = (
                        "/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage2_Evaluation/"
                        f"Resources/scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_{samp}_{class_}.tsv"
                    )
                    motif_match_df, motif_count_df, motif_enrichment_df = compute_motif_enrichment(
                        mdata,
                        prog_key="cNMF",
                        data_key="rna",
                        motif_file="/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage2_Evaluation/Resources/hocomoco_meme.meme",
                        seq_file="/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src/Stage2_Evaluation/Resources/hg38.fa",
                        loci_file=loci_file,
                        window=1000,
                        sig=thresh,
                        eps=1e-4,
                        n_top=2000,
                        n_jobs=-1,
                        inplace=False,
                        gene_names_key=gene_names_key,
                    )

                    motif_match_df.to_csv(os.path.join(output_folder, f"cNMF_{class_}_pearson_topn2000_{samp}_motif_match.txt"), sep="\t", index=False)
                    motif_count_df.to_csv(os.path.join(output_folder, f"cNMF_{class_}_pearson_topn2000_{samp}_motif_count.txt"), sep="\t", index=False)
                    motif_enrichment_df.to_csv(os.path.join(output_folder, f"cNMF_{class_}_pearson_topn2000_{samp}_motif_enrichment.txt"), sep="\t", index=False)
            """

        # ────────────────────────────────────────────────────────────
        # 5g. Explained Variance
        # ────────────────────────────────────────────────────────────
        if Perform_explained_variance:
            print("Computing explained variance...")
            compute_explained_variance(
                cnmf_obj,
                X,
                k,
                output_folder=output_folder,
                thre=str(sel_thresh),
                program_name=mdata[prog_key].var_names,
            )

        print(f"Done. Results saved to {output_folder}")

print("Pipeline finished.")


## Motif Enrichment (Optional) — scE2G preprocessing snippet

Keeps the previous one-time helper for building enhancer/promoter input TSVs from `scE2G_links`. Not part of the standard evaluation loop.


In [ ]:
# # Format files

# Thresholds
# score_thresh_abc_e2g_enhancer = 0.015
# score_thresh_abc_e2g_promoter = 0.8

# for i in range(4):
#     e2g = pd.read_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}.tsv'.format(i), sep='\t')

#     e2g_enhancers = e2g.loc[(e2g['class']!='promoter') &\
#                             (e2g['ABC.Score']>score_thresh_abc_e2g_enhancer)]
#     e2g_enhancers = e2g_enhancers.loc[:,['chr', 'start', 'end', 'name', 'class', 'ABC.Score', 'TargetGene']]
#     e2g_enhancers.columns = [        'chromosome', 'start', 'end', 'seq_name', 'seq_class', 'seq_score', 'gene_name']

#     e2g_enhancers.to_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}_enhancer.tsv'.format(i),
#                           sep='\t', index=False)

#     e2g_promoters = e2g.loc[(e2g['class']=='promoter') &\
#                             (e2g['ABC.Score']>score_thresh_abc_e2g_promoter)]
#     e2g_promoters = e2g_promoters.loc[:,['chr', 'start', 'end', 'name', 'class', 'ABC.Score', 'TargetGene']]
#     e2g_promoters.columns = ['chromosome', 'start', 'end', 'seq_name', 'seq_class', 'seq_score', 'gene_name']

#     e2g_promoters.to_csv('scE2G_links/EnhancerPredictionsAllPutative.ForVariantOverlap.shrunk150bp_D{}_promoter.tsv'.format(i),
#                         sep='\t', index=False)
